# Enterprise Document Assistant: GPU Batch Indexer & Benchmark
**Notebook Google Colab (Free T4 GPU)**  
Notebook ini digunakan untuk menjalankan proses embedding berkecepatan tinggi pada GPU untuk dataset korpus perusahaan dan benchmark BEIR FiQA.

In [ ]:
# 1. Install dependencies
!pip install -q rank-bm25 sentence-transformers pydantic pydantic-settings pyyaml tabulate pandas httpx
print("Dependencies successfully installed!")

In [ ]:
# 2. Clone atau Clone Repository ke Colab
!git clone https://github.com/beambomb/Document-Assistant.git
%cd Document-Assistant

In [ ]:
# 3. Verifikasi GPU Availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 4. Jalankan Pipeline Ingestion & Chunking
from src.ingestion.loader import DocumentLoader
from config.settings import settings

loader = DocumentLoader()
docs, chunks = loader.load_directory_markdown(settings.raw_data_dir / "id_enterprise")
print(f"Loaded {len(docs)} documents, created {len(chunks)} metadata-enriched chunks.")

In [ ]:
# 5. Batch GPU Embedding dengan Sentence-Transformers Multilingual
from sentence_transformers import SentenceTransformer
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(settings.embedding_model_name, device=device)

texts = [f"{c.title} {c.section} {c.text}" for c in chunks]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"Generated embeddings matrix shape: {embeddings.shape} on {device}")

In [ ]:
# 6. Jalankan Evaluation Harness Lengkap
from src.evaluation.harness import EvaluationHarness
from src.evaluation.reporter import EvaluationReporter
from pathlib import Path

harness = EvaluationHarness(dataset_name="id_enterprise", use_mock_embeddings=False)
results = harness.run_ablation()

report_md = EvaluationReporter.save_report(
    rows=results,
    output_md_path=Path("reports/colab_gpu_ablation_results.md"),
    output_csv_path=Path("reports/colab_gpu_ablation_results.csv"),
)

print(EvaluationReporter.generate_markdown_table(results))